In [1]:
import os

from dotenv import load_dotenv
from langchain_aws import ChatBedrock

In [2]:
load_dotenv()

True

# Connect to Bedrock...

In [3]:
# llm.py
llm = ChatBedrock(
    model_id=os.getenv(
        "BEDROCK_MODEL_ID"
    ),
    region_name=os.getenv(
        "AWS_REGION"
    ),
    model_kwargs={
        "temperature": 0.2
    }
)

# Load Document

In [4]:
# ingest.py
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_aws import BedrockEmbeddings

from langchain_pinecone import PineconeVectorStore

from pinecone import Pinecone

C:\Users\m\AppData\Local\Temp\ipykernel_13608\3967783626.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


ModuleNotFoundError: No module named 'langchain_pinecone'

In [5]:
loader = DirectoryLoader(
    "knowledge_base",
    glob="**/*.txt",
    loader_cls=TextLoader
)

documents = loader.load()


print(
    f"Loaded {len(documents)} documents"
)

FileNotFoundError: Directory not found: 'knowledge_base'

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)


chunks = splitter.split_documents(
    documents
)


print(
    f"Created {len(chunks)} chunks"
)

In [ ]:
embeddings = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2",                 # change this model
    region_name=os.getenv(
        "AWS_REGION"
    )
)

In [ ]:
pc = Pinecone(
    api_key=os.getenv(
        "PINECONE_API_KEY"
    )
)


index = pc.Index(
    os.getenv(
        "PINECONE_INDEX_NAME"
    )
)

In [ ]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)


vectorstore.add_documents(
    chunks
)


print(
    "Knowledge base loaded into Pinecone"
)

## Create Pinecone Retriever

In [ ]:
# rag.py
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":5
    }
)

In [ ]:
# database.py
from sqlalchemy import create_engine

DATABASE_URL = (
    f"postgresql://"
    f"{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"{os.getenv('POSTGRES_HOST')}:"
    f"{os.getenv('POSTGRES_PORT')}/"
    f"{os.getenv('POSTGRES_DB')}"
)


engine = create_engine(
    DATABASE_URL
)

In [ ]:
# tools.py
from langchain_core.tools import tool

from app.rag import retriever
from app.database import engine

@tool
def search_knowledge_base(
    query:str
):
    """
    Search HealthSecure policy documents.
    """

    docs = retriever.invoke(
        query
    )


    return "\n\n".join(
        [
            d.page_content
            for d in docs
        ]
    )

In [ ]:
@tool
def get_member(
    member_id:int
):
    """
    Retrieves member insurance information.

    Use this tool when the user asks about:
    - their plan
    - deductible
    - copayment
    - coverage details

    Requires:
    member_id
    """

    sql = """
    SELECT *
    FROM members
    WHERE member_id=:id
    """


    with engine.connect() as conn:

        result = conn.execute(
            sql,
            {
                "id":member_id
            }
        )


        row=result.fetchone()


    if not row:
        return "Member not found"


    return dict(
        row._mapping
    )

In [ ]:
@tool
def get_claim_status(
    claim_id:str
):
    """
    Retrieve claim status.
    """

    sql="""
    SELECT *
    FROM claims
    WHERE claim_id=:id
    """

    with engine.connect() as conn:

        result=conn.execute(
            sql,
            {
                "id":claim_id
            }
        )

        row=result.fetchone()


    if not row:
        return "Claim not found"


    return dict(
        row._mapping
    )

In [ ]:
# agent.py
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

from langchain_core.prompts import ChatPromptTemplate

from app.llm import llm
from app.tools import (
    search_knowledge_base,
    get_member,
    get_claim_status
)

In [ ]:
tools=[
    search_knowledge_base,
    get_member,
    get_claim_status
]

system_prompt = """
You are HealthSecure AI Assistant.

Your role:
Help HealthSecure members and non-members.

Rules:

1. Use tools whenever required.

2. Never invent information.

3. If required information is missing, ask the user.

4. Use:
- search_knowledge_base for insurance policies and benefits.
- get_member for member-specific information.
- get_claim_status for claim-specific information.


Examples:

User:
"What is my deductible?"

Assistant:
"Please provide your member ID."

User:
"Why was my claim denied?"

Assistant:
"Please provide your claim ID."
"""

prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        system_prompt
    ),

    (
        "human",
        "{input}"
    ),

    (
        "placeholder",
        "{agent_scratchpad}"
    )
]
)

In [ ]:
agent=create_tool_calling_agent(
    llm,
    tools,
    prompt
)


executor=AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [ ]:
# test agent
response = executor.invoke(
    {
        "input":
        "Why was claim CLM1003 denied?"
    }
)


print(
    response["output"]
)

In [ ]:
# memory.py
from langchain.memory import ConversationBufferMemory


memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

In [ ]:
# agent.py - updated
from app.memory import memory


executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)